# Z-Image-Turbo on Kaggle T4×2 GPU — Production REST Demo

**English**

This notebook is the reproducible Kaggle entry point for running `Tongyi-MAI/Z-Image-Turbo` as an **API-only** image-generation service on exactly **two NVIDIA Tesla T4 16 GB GPUs**. One resident BF16 `ZImagePipeline` spans both GPUs: the transformer is placed on `cuda:0`, while the text encoder and VAE are placed on `cuda:1`. This is one logical pipeline, not two independent replicas and not a 2× speed claim.

The workflow validates the attached Kaggle model, reconstructs the audited server payload, starts an internal worker plus an authenticated FastAPI coordinator, and executes deterministic acceptance checks for the qualified `safe_512` and `high_768` profiles.

This is an independent engineering qualification project, not an official Tongyi-MAI release and not an SLA-backed hosted service.

**Tiếng Việt**

Notebook này là entry point có thể tái lập trên Kaggle để chạy `Tongyi-MAI/Z-Image-Turbo` dưới dạng dịch vụ sinh ảnh **API-only** trên đúng **hai GPU NVIDIA Tesla T4 16 GB**. Một `ZImagePipeline` BF16 resident trải trên cả hai GPU: transformer nằm trên `cuda:0`, còn text encoder và VAE nằm trên `cuda:1`. Đây là một logical pipeline, không phải hai replica độc lập và không phải tuyên bố tăng tốc 2×.

Workflow xác minh Kaggle model đã attach, khôi phục server payload đã audit, khởi động internal worker cùng FastAPI coordinator có xác thực, rồi chạy các acceptance check xác định cho hai profile đã qualification là `safe_512` và `high_768`.

Đây là dự án kiểm định kỹ thuật độc lập, không phải release chính thức của Tongyi-MAI và không phải dịch vụ hosted có SLA.


## Before you run / Trước khi chạy

**English**

Before running this notebook in Kaggle:

1. Select **Accelerator → GPU T4 x2**.
2. Attach Kaggle Model `dangkhoa2016/tongyi-mai-z-image-turbo`.
3. Select **PyTorch / default / Version 1**. The expected mount is `/kaggle/input/models/dangkhoa2016/tongyi-mai-z-image-turbo/pytorch/default/1`.
4. Keep **Internet = ON** only if you want to exercise the optional Cloudflare Quick Tunnel step. Local qualification uses loopback HTTP and does not require the tunnel.
5. Use a fresh Kaggle session and run the notebook from top to bottom.
6. Do **not** copy the full model into `/tmp` or `/kaggle/working`; the model stays read-only under `/kaggle/input`.

Qualified public profiles:

| Profile | Resolution | Steps | Guidance | Role |
| --- | ---: | ---: | ---: | --- |
| `safe_512` | 512×512 | 9 | 0.0 | default deterministic acceptance path |
| `high_768` | 768×768 | 9 | 0.0 | qualified higher-resolution single-flight path |

Unsupported by design: FP16 fallback, CPU fallback, 1024×1024 generation, multiple concurrent GPU pipelines, and full model staging to `/tmp`.

**Tiếng Việt**

Trước khi chạy notebook trên Kaggle:

1. Chọn **Accelerator → GPU T4 x2**.
2. Attach Kaggle Model `dangkhoa2016/tongyi-mai-z-image-turbo`.
3. Chọn **PyTorch / default / Version 1**. Mount mong đợi là `/kaggle/input/models/dangkhoa2016/tongyi-mai-z-image-turbo/pytorch/default/1`.
4. Chỉ cần giữ **Internet = ON** nếu muốn chạy bước Cloudflare Quick Tunnel tùy chọn. Qualification local dùng loopback HTTP và không phụ thuộc tunnel.
5. Dùng fresh Kaggle session và chạy notebook từ trên xuống dưới.
6. **Không** copy toàn bộ model sang `/tmp` hoặc `/kaggle/working`; model được giữ read-only dưới `/kaggle/input`.

Hai public profile đã qualification là `safe_512` (512×512) và `high_768` (768×768), đều chạy 9 bước với guidance 0.0. Chủ động không hỗ trợ FP16 fallback, CPU fallback, 1024×1024, nhiều GPU pipeline chạy đồng thời hoặc staging toàn bộ model sang `/tmp`.


## Workflow map / Sơ đồ quy trình

```text
REST client
   |
   v
FastAPI coordinator :8090 (CPU, Bearer auth)
   |
   | bounded single-flight queue
   v
Internal worker :8101 (loopback only)
   |
   +-- Transformer ---------------- cuda:0
   +-- Text encoder ---------------- cuda:1
   +-- VAE ------------------------- cuda:1
   |
   v
One resident BF16 ZImagePipeline across T4 x2
```

**English**

The notebook intentionally separates **runtime qualification** from the optional **public transport** check. A temporary tunnel failure does not invalidate a healthy local model/runtime.

**Tiếng Việt**

Notebook chủ động tách **runtime qualification** khỏi bước kiểm tra **public transport** tùy chọn. Lỗi tunnel tạm thời không làm mất giá trị của một local model/runtime đang hoạt động đúng.


## Step 1 — Runtime and model preflight / Kiểm tra runtime và model

**English**

Verify that the live Kaggle session exposes exactly two Tesla T4 GPUs, enable the qualified 4-worker Hugging Face parallel-loading path before Diffusers is imported, and confirm that the attached model resolves to the expected `ZImagePipeline` before any service is started.

**Tiếng Việt**

Xác minh session Kaggle hiện tại có đúng hai Tesla T4, bật cơ chế Hugging Face parallel loading 4 worker đã qualification trước khi import Diffusers, và xác nhận model đã attach đúng loại `ZImagePipeline` trước khi khởi động service.

### Expected evidence / Kết quả cần thấy

**English**

`PREFLIGHT=PASS` plus the printed Python, PyTorch, CUDA, Diffusers, Transformers, Accelerate, GPU/model provenance, and parallel-loading configuration JSON.

**Tiếng Việt**

`PREFLIGHT=PASS`, kèm JSON provenance/runtime in Python, PyTorch, CUDA, Diffusers, Transformers, Accelerate, GPU/model và cấu hình parallel loading.


In [ ]:
from pathlib import Path
import json, os, platform, subprocess, sys
os.environ['HF_ENABLE_PARALLEL_LOADING'] = 'YES'
os.environ['HF_PARALLEL_LOADING_WORKERS'] = '4'
import torch, diffusers, transformers, accelerate

MODEL_PATH = Path('/kaggle/input/models/dangkhoa2016/tongyi-mai-z-image-turbo/pytorch/default/1')
WORKSPACE = Path('/kaggle/working/Z-Image-Turbo-Kaggle-T4x2-REST-Server')
EXPECTED_MODEL_INDEX = MODEL_PATH / 'model_index.json'
EXPECTED_GPUS = ['Tesla T4', 'Tesla T4']

assert torch.cuda.is_available(), 'CUDA is required'
assert torch.cuda.device_count() == 2, f'exactly 2 GPUs required, got {torch.cuda.device_count()}'
gpus = [torch.cuda.get_device_name(i) for i in range(2)]
assert all('T4' in name for name in gpus), f'T4x2 required, got {gpus}'
assert MODEL_PATH.is_dir() and EXPECTED_MODEL_INDEX.is_file(), 'attached Kaggle model missing/wrong variation'
model_index = json.loads(EXPECTED_MODEL_INDEX.read_text())
assert model_index.get('_class_name') == 'ZImagePipeline', model_index.get('_class_name')

runtime = {
    'python': sys.version.split()[0], 'torch': torch.__version__,
    'cuda': torch.version.cuda, 'diffusers': diffusers.__version__,
    'transformers': transformers.__version__, 'accelerate': accelerate.__version__,
    'gpus': gpus, 'model_path': str(MODEL_PATH),
    'upstream_revision': 'f332072aa78be7aecdf3ee76d5c247082da564a6',
    'parallel_loading': os.environ['HF_ENABLE_PARALLEL_LOADING'],
    'parallel_loading_workers': int(os.environ['HF_PARALLEL_LOADING_WORKERS']),
}
print(json.dumps(runtime, indent=2))
print('PREFLIGHT=PASS')


## Step 2 — Materialize the audited API-only source / Khôi phục source API-only đã audit

**English**

Reconstruct the exact embedded server/test payload in `/kaggle/working` and verify both the archive digest and every allow-listed file before writing it.

**Tiếng Việt**

Khôi phục chính xác server/test payload được nhúng vào `/kaggle/working`, đồng thời xác minh digest của archive và từng file trong allow-list trước khi ghi.

### Expected evidence / Kết quả cần thấy

**English**

`BOOTSTRAP_FILES=<count>` followed by `BOOTSTRAP_INTEGRITY=PASS`.

**Tiếng Việt**

`BOOTSTRAP_FILES=<count>`, sau đó là `BOOTSTRAP_INTEGRITY=PASS`.


In [ ]:
import base64, hashlib, io, json, shutil, tarfile
from pathlib import Path

WORKSPACE = Path('/kaggle/working/Z-Image-Turbo-Kaggle-T4x2-REST-Server')
PAYLOAD_SHA256 = '3297e31dc53177e547d09b3d33febc2ebb46aba305398272dc9899d6528e3cfe'
MANIFEST_SHA256 = '5fbc29b146223409db3f97dc6619a6e68df78a386a4cd3c5689406c4e44bcb66'
MANIFEST = json.loads('[{"bytes":13,"path":"src/zimage_server/__init__.py","sha256":"75ad4f4c2943325f88064ad2beae890b59674be2bf0404b30afc5584dc4de4bb"},{"bytes":4913,"path":"src/zimage_server/app.py","sha256":"cd91cc210e24c8fc1cab1cf3ef11dfdd8bb4eb9dc4dd82f5413aac92bb882c8b"},{"bytes":347,"path":"src/zimage_server/config.py","sha256":"eed00972bfc315774cd2d71db20935ce1c6f0048d58fdcd2dc9137ba58b94a72"},{"bytes":928,"path":"src/zimage_server/internal_client.py","sha256":"ab65fd699dc0c04d9f58a3686cc8732206a6a8a7cc3210fc85bd97e8a17d3606"},{"bytes":3128,"path":"src/zimage_server/jobs.py","sha256":"ef3b335d6d129e9fbcaa77cd3716e329d009fdb67c0074121bf7ee92ec21c85b"},{"bytes":4296,"path":"src/zimage_server/runtime.py","sha256":"675eafe52416c680e0a2f46474ba51c8eedc251aca5cc443c292c41110fe86d3"},{"bytes":776,"path":"src/zimage_server/validation.py","sha256":"7355f1a3121f1e664f5fb0cf1422a0e20c8ad02f978d5959a5186a1e81692922"},{"bytes":2258,"path":"src/zimage_server/worker_api.py","sha256":"18006dfecc4fbe0a38044ef9342e2974f72455eecc998eb6d568733897f62cff"},{"bytes":4223,"path":"scripts/endurance_queue_acceptance.py","sha256":"4e77a3ac8e8eecfa70f9316e8fc931334f9d15a63e560345177e57f31b211308"},{"bytes":1847,"path":"scripts/error_boundary_acceptance.py","sha256":"8ef6edeb6041f54d9496f804462b957a3c94cd2439ba06b0815891fe3bdf70f0"},{"bytes":2449,"path":"scripts/finalize_evidence.py","sha256":"5b9787a04f05e904af985394e5183cc14f132ac36a63946111ccf71703d5cda0"},{"bytes":2475,"path":"scripts/golden_rest_acceptance.py","sha256":"ef72c5cda5e45a1330c4c835b55bf791cbd718365e399f928f89290fe2c92796"},{"bytes":1153,"path":"scripts/local_control_acceptance.py","sha256":"de226083874f29f42f1cca90ff7c7a9ed60f143ff6992b43a1622214a77350aa"},{"bytes":4394,"path":"scripts/mixed_profile_acceptance.py","sha256":"d233634d294e098d01876331bcebff11e93c406311980bbf63c9adea2c89ee37"},{"bytes":2624,"path":"scripts/public_tunnel_acceptance.py","sha256":"ce8b0ede3fc8e31f0b38fa02808fe9eb5ba17adfeafa9eb4ac29bf582122cc22"},{"bytes":1999,"path":"scripts/qualify_mixed_existing.py","sha256":"fe6e1802288b7e92161593a8ae11faf55009512d189e07285bccd7eacc5fb14c"},{"bytes":2215,"path":"scripts/start.sh","sha256":"f40f48a7d49599545afc3e50d58a33450fd901151c76c5aaa5f3c1a760a7d972"},{"bytes":403,"path":"scripts/stop.sh","sha256":"6075f5c3dd4c7591fb0e234a0d3b0ec178bd294d3b957bdd1fd91cea8fe2daef"},{"bytes":264,"path":"scripts/status.sh","sha256":"adc8bdfddab5f53550151aeb5fd57980d4ced498c9382bcdd8e57d2c45d9a788"},{"bytes":1022,"path":"scripts/start_tunnel.sh","sha256":"6e85fd13630818a2161d7a3265128e2cf264bfbc6d9a28bf5b820238ccb8d966"},{"bytes":1118,"path":"tests/test_api_jobs.py","sha256":"d4c276f9d8fa223fd37722d942e7f595ec96c03cfda9cc58ecedb5a29836d7fe"},{"bytes":1214,"path":"tests/test_contract.py","sha256":"a415472193557627d315154f76aa8a0cb0bec74063f8de776f574b39202dadf9"},{"bytes":1132,"path":"tests/test_jobs.py","sha256":"c2b664ff7e0e1973901f3b2b436fdd293ac205a799a8d7d2735eae662ef60777"},{"bytes":488,"path":"tests/test_profiles.py","sha256":"70d0ca794d43ba888aa760810fcf8c401a87398e12c080bef85c12d854a6fb53"},{"bytes":759,"path":"tests/test_root_metadata.py","sha256":"8bbeb036fbe0eab31e083ff6e120788fc8ed16e74d34c21fb42dcc2b2f25603f"},{"bytes":920,"path":"tests/test_validation.py","sha256":"7d290a312d25d13e3ae2cbca06f2373ab45adbada110ef51a9e91086059274fc"},{"bytes":39,"path":"pytest.ini","sha256":"0922d82a60b616ffaf801784738325d7f1925ec773c99135f64767d14166d725"},{"bytes":689,"path":"pyproject.toml","sha256":"33105bcd4544e466fbb671dc32edadca7b5e61c23a5e7fd795e3f517461bf4dc"},{"bytes":3871,"path":"scripts/api_contract_acceptance.py","sha256":"2c33e8ea927c67eaa5e3a3a51c1c59c3282f13782356e8cb3c420506e9fc8a61"},{"bytes":5259,"path":"scripts/verify_evidence.py","sha256":"bf1394291d78d4bd98c34b6ad34c9dca0c390d9a42125db6ef66692242a7d201"}]')
PAYLOAD_B64 = 'H4sIAAAAAAAC/+1923bbRrLoPOMrEJyZEemQFEHdbE7oNbJNO5rIkiLLmZ2hubEgEhQRgQADgLIUjZ7OD5yP2h92qqov6AbAi2xZyfaQy8sC0NXV9+qq6qrqJB5s/uZP3AvPSbz4yos3HccP/dRxGtObPz3Mrwm/3e1t+gu//N9m097JnvG7vbXT3PqT2fzTI/xmSerGUPyf/jN/juMGgeOYHbPXN/60/v2n/ZLC+nen04db+iusf9ve3rVz67+1s7Ve/4/y8yfTKE7N8cQdGPw5SoxRHE3MqZuOA//c5J9P4JUlDKIw9a5TJc1NbsIB/zxxQ5hNscFgRy7079QXgK+8qRcOk5r5Gr7vnxzUzO89d+jF8Pfs7KR7PfCmqR+FWt5G7CXTKEy8RGB57QfeKf/I63ozdMPUHwiIF27ivY2GXgAl+V4wrJkj/ONcuYE/dNNIVK8BlR75F7J63df77w/PnJPT49cHh92ayR/ecehfonNZiX9E529ZU2vmjzNv5r2eBUE3jhG3MQjcJDHfeKEXu9igU+/XmZekFVmvatsw4TcFtNO0bSZpDCSYqlqZ+KETeOFFOu7YNXPiXou37eaz3SplSzxv2Db9MIVM2y2BaQTdIlDlWmIQzN9znVCxWPkWw/p3qvXES8fRkD4MvZEZRqFzHrjhpcNgK4MAhg9QzFhZVbP+HP+y9uDPx0wpA2lAij+tVLNU/MWun3jmTwhAHSbqYU5gNVLmc8+kQnnNKJOXzuKQoV3QHOyERe25DKOPocMBV2oMJVOt/FBOiGUNmoVUkKnXqNgOA+vkuLN0XMH/otj/jSYMG8d/m0dR6MFwslVSAWB3FqQd/FqlyuITq4t3PfUGqTcE6ChpeOGVH0dh48JLK9a/Dt7uv+k6sN6cs+MfukdWzbR4jfhgaWWbUVz82EA6mSYffaip9cJzYy82LWVcWRdoq7gCWdJZ4gxgwsPkhck89FLXDzrWxE8SP7yAYSY8aXTphbxCyWw6DXxqhlZ+b6/dV2ssm8srixQMVvNkChidoX+Bq03gqknoe1V4K6uwH9Isy1eYDd8g9tzUc2DjrDypmdRPzscovvTitnkeRQE05SyeeTUznoVAEGj04AWG1A+9JOHvuNB/RTriJP5vnlzdNMqcWrZFB6hlmG445JhNP1EmBP4Y2QJUXhy6gTOA3gC0nIJBJudCkqiayfA5WLGbrJsY5k4OWpnOvBkIIp9hUDRsBM53BgDMaGeF9wn7k++Fjv5aLbTdC2AMscmcIpRtRJhA32n9B/4Idg43rMBwtUXHKtPiBmmKuvxFraFvcZ7p/as0qzEIosSrVFlNADsSdIa+kvpp4HWsf9UPkNOqn83i88j8wb24CDzzbPu6ZZ52352Z74gBg8UJ/yfQxx2r2bAbTfggKt0RD1XRXmDWaIlvApQfDoLZ0AP5zUkGY2/idl670D9VSfviKEodoIcukEtXJcqcJt1q7bKQIfQHntU2rd/qxCPWU6x5/ZJqXk+h5lZNz5PeTCkD7NlpHTbvfDpvGoLwxunpw2iQYOImPeQSx54bpGNK5o85AJpslM6ecsl+OIoo9crepOcs/a7QobyErPfYh7JusxjdQNTRpVVExSsDWJD78cKB7yWdHueFKkT9q31lmBBcLUefhcUZuIyU7aikjK8cnMu0lLyhsjn5I2URl5cl2iz6mpE2jha7wA9/IVoLHSEyYWU8lUBUqnl2gUBYfzHM1U9vI+Eq7Lj0tTA2ciqsOjoIrQ4O3+GR/t2G7gQo95UbJxX+uWqOcIOC7zUBqbIRDT/1JtAbWVfx4dH6qjAsghAx+peNRQmtuiuZrSuta1NwUzivxaM60OxBne3TKBFdiliTzWy/SJZ1MG2ccjRbzVbW4xyLV4kZF90uMtaPsFj4rJy7VOSY4H4sMvABfKB5vag6aXyjIwZZxfGRjRL7UzI7n/ip6MTGBMUQZzibAHueofGoEjmBBrZPTLhfxbdbz5QFGVcAAbCsxJDAY3FW+kOYU6zSMMsyikrb/1CZj/BRtIF/QXiQiJQEfL0rW+sowW3esmLuVl/0kKHCMjE54XeYbkAolMHEBrEKaTOSoD6pHtvN7awegJkqMYpm4bAou2ApSzuX0YB70FUEX3eyLKEn1kDf/KZjWijaBF7qWULgQRhBVxIQCu9LWLabz4p1kaVk1UE1EPQJqn8qrF68vD4rPZql01nqIBiTKwtbOyY1/MQhibv6GT3GijK59Ficl6peqIKlgjzhDX3XQa4UhDicYZtTyInqoMDDXblDlcMnzlJzVDCvQbpjXLwi31WNr17/yzRiD6kCXnb+g8re3PlPy95d638f40f7MYqDpCvLVKzyE6yCv8uXCoD/5oUd5PmrXMl5whWPtHo++sN0TLoLeh17/sU4zd6T1Jsm2evFzB+64QCm3sBF1eUoiNwUCswpL2EJWok78pwdu2UZgnVGfttgcqpIa4vKVKgeHfhW43Vgz1R+B+ieXnKn2WhWmSxojQHa2dt9WkAG3yQyel6M7M74X7r+c8qiByAES9a/vbvdzK1/GKz1+c/jrX/HGc1g1/McRx7ohLBxM8HNEOdCvyRRKJ5ncRD45w2Pjjv0b5wFN4x/Hp/+0D11Xuy/oxU8TtNpe3PTbu01YIU07PZTu2lbXIWqKgsrpPIc+oOU0RRNskH9c66kBrxGwFRW1AK/zXRAZupPPGAdOjtVFGTEaVKpWgPb2AAqNKwIsKqhiEWSRylIRAW1CKnegB+iLoL3kXWLXAiJQQ3HQY7Dce7a5i18uEOdEfWDrmRF5rNNPZHrEt5y6NZcV4hzJlmvfJ8IWVpRfSF171DDURhMsNBqA9h1YL8q1iwd1Z9a1Qx6TCcRSefWeolq1jCtn3GVn4vK9gFVfBPRWXdZLnYI07FOjt+d8aKr9xtc/p6N5laz+dnjqU7jBjGhpeLueTS8gb6Gbw0cXpDnh57aPTWTMCQdGP1p4A604x5icE9nIdabHQ6NhAiCJdIEaCA2nAxY0p2liMmPT/9RiHtYA4Al9H+3uZen/609u7mm/38w+k+KGPGSjpl660J+gOktdwJgiISpwPRGspSeN8V3lgLEEE8AedpLNwjc88CTR+e6Aqqirh9xdM6YUwaeHSi1pUJBGLJVEi8YicO3tiyq10N62q+Z/M+887eMDCCehqMeiIX8eClLJQxt1lcNakQvnYFI3UvSmBfFT3X7gEIBUzTiDJM7JDkX9ppO1tuNF6gs8IbvvIk7HUexV9GrDTTeziPCBc12EbUSwD3f5QCDaHCpFXYIHwr1Yska3Bk9VWAhXXhpRyCLpjVSfHcsRmDqUJE61x5DLVxvEglJoqwEduAsztNwQLkCkw3n1L1Buq5sj9q5vdR+gtR/4VjQLzgpG/jfNtDvsXctQcPoI7YGplcD/6uoaoVBFCOSVbSTrHapN3TcFFIAK6osWS3hXcz+Cv9UrZlCkdI22Qmw5BV0vT3titkI6ftcNsQ9Vrs+HR1gvQsnLPq0ariDX2c+zKBzxArDyE8LdfwLC9cr0JhG04roIrJQKNH05Na1JTgZ6kdUqo0gzaqWraoGqpmgW12YA7KcQrdWC1ohBmooxwliDqnKRsFg8dXZXq37ucJQ6YQynaFSGVlfzFmV2sTcOTaxxK5fUk/J/3BpOccYsszuMPDJbIQmNSyzKI1CfwCM9bcie9a8MR5JFQC/k2jmN3heUxeqSLOp8IN3wybBAhyKEtRH7lrqJ+VquStBr2ip1e/UyCSAEaiABGLnebQz1jN6neRwsNN9bHfO7qb08GXOtNVXBWdCGRSuCR0HbXCllPGXyA8rgg9uKZUkqjuvkmykkeAumMOsMjiy9xxTNXvqJpfOEMBySLLBKTmpkiQd6lIYuiwJ57pc7GUwPUFTAKeF1JCvf20m5MdrJVJXTm+VKQq7DTIFqJu+T+aYbxr9OdtQ1nW4XcihYvxHeV88WHvkgls59wi4rmS8Wovm4RAnDLSX4aOWc6kw/qA9wMjMozafFUk8mjCoKeoOgAJOvCTBMzbaGSj1TisD6gLM7s28xZqxArEXeG5SVsV5y3ptOf+1nv/ETNR6QBXAEvv/5padl/+3tlvr858/mvw/dhP0ByhTB5PwP9dpQEJF8YC7EAz90WgGE06eN/2LLCNP/KmHPGe5dX7OGJ/bfeMGwAH4F8+hCW0Yb49fdQ+dk/2z78UBtrXJbJw2/RDYsk2ygUk2h254cTmO3FbT3t0EDvjixq9PXL+uWUdtTm+oAZvcCHvTBjHl+P3Zyfsz5/T4+KxQBIq6wApsalafdWb1WUerzzpafdaZ1ecmO9tOAOfb7tn+q/2z/YfAKow9AW33v066L8+6r5xX3Z8OXnadt/snbIOJ3TAZRfGEjLqasK2g8azDFMD4yYZPVy7uMvadVLiw8eJqmTk6l7zqZAqjC0WSmKMlIP8C5AdGe5iUAgw9NFhzJu5UJMsSp7E3CvBQLmN7dXGIS780do3BbOiiKYJ75fqkBQJRJ4rVRF7UIIKmkRzUKjNY0PS51puT987Z9n+1nNPuj+8PTruvFAEWN2tsVE8pA9hrh5eDyRWfWQn6KOPAaFx4lVa1rzbADW8q1tm2JbwQMJc0LBQfkurSmo6KVW0zq8XkzirYbWTLB7ts6MesszCpoiytTWBEyJbMD4fedYNU/9XF9h569zFcB0e4kt4evHt3cPSmaNtxa11MZ9RdqHehvzVRMNmeMCYoqxcqWIbCHPmchGV7VxjnkiU4HgvMmTRyVgluO5tmVdWqNWYOF2zD9OIRmzfAlVdLp75O5RpIxxzADEsQXocVXYTMtUZLpIZ12JQSbctByCXTsc7dAM+nhzl7aFQdTrxJFN90bpvQSfb2G/8FSNa28mwNpjPswNZTfL/TEQBT7QY0zIkThcENafMykOrCVV7SZ2Zd9OmC5U8CoOzUxnjkZMnaFM5n/aZjltDAFdbM+6OyfOZtroC7kkn75ImcOHK2qv2AszbfNzhvJVIBoBRjFA1jpZqo9LRQ7Q+aiAts18pW5tHxmXN4vP9Ko2vCmLkjt2WUX3rSSLPfVzpDyONMGtcl9MxXJU8C9fop9BNkQqCgU8+95PPXQYEtATqaHdl5gM6TsjIDU1Yl77oIFfkM8xvxpcL6umNhWe0m0LKJG85goqNRacUPSenVYxan/ep9yQFxFJKqwGjoq545wnVEV6JzXl9fdNwEhfdzg73WckpbNFgREPSmA4SzCVDrkRd7ZLZChiwCnN508JyBi4DUP+eyiN7syCeFMDSYfXivqexyWXXuSyJw7AFa5/4q9L80uue9YOrdlg2Lwsg1Jpe416EjWZgmjKaZ3rUP8zW6zB1YaLzafTIqVpNQdZWN3ERTgWyB3DWmqjqJWtVI3CuvoqCo4eKZuGnHOtH2TuYHBwVw3r2RjN3Wzq6alU60nfOb1EsqVToY4c5zGRZ3lNKB15ylJNVSt3lnHLnI2woNyPnkZEbdOvmo5X2RyL47v/pyUDTImbtC2dS32MgrQGUryKJVoACVrApLn/8K8KKFUcKTFL2U9JVA7c59y1eXxhWPmWjwcqmqiS5jk5Qv+T6c+tewSdGiImD4m4PgNJeRWIBhD+VANHcAhv7me2LBLqd6Zomnii4ZFRYK5zw/xj4QARRiKopVC5ul6CQ3hNXZaQF7SCIOHnwJK5f89s11j8zSHXvCH+S8ieVqKO64pT51I+Rib/07aPBtQddmuQFyVKmc6cqWJ7pTQMBelxs4vuZIizMfgQCYk5/21EXVkCzj0qoQqgX1yTDNr1NOnZlnEJQ5stbkPZT+L1OrPJQKcJn953ZB/7fd3Fvr/x4z/gcwhNMbPEcJp9wSsZybEn70TmborXxUzL1zBNEfCc4FjVRAEqvomApICo76SnCFkTULpfv/EHjEEA8xEiCpWRlCGHNj5FzCacNN4NG9YQ1BxSLM9bRinb55ITxWuNYDYP0ET3FAroIs1QYQusqi6lhhFNZZBpM2z0SEMkhRGKBdHjEBDzEUrnaz0Id9C9jaIIqRd0XiCAWzzwQMRHHsgnRQt2vmFmxWLvCRnWa1QV+Bca5msQCG5ncd0240UTmjI8bvsKIWVX7oCSmSjRCve2H3krxVbuCMImOVG0oFZOKHfCfAJsJbRaX4FuwJarJ7nUv2XMyedSh+0EGgM4hpUbhMS+sSSNXeGdx/7v5RpP/cFBsD7zwO/d/earXy9N+21/b/f7Tznyi5t/HnJ8eJysJDFcNCzY31JE6N+AmmfrTEFVuG4Yjkjp4CEq0j4hSoER1I3eCT06BmoAev3LSPvymWHPKbag0g7Prm2lmKs5V/0gpcIWSUHguqEAbK0JVvuZBC7BjF4GZD7lD0i9jrWGc0ZtMhaRipPzrUZKiyQ6pMYfjOI9hgGn/M+qKjGECg26Xsj05mDKWZ47Bye6xz0QZCVIzZ8VcLgJr5DXlfrOayUZY/Z+whQD7XIKNgiCEQL7IW4WPD4t3wPnBYH3DHmGWWuNq45oxxaQDrCFBujpszwlV4ApaCfoFly3m1SDulrTKy+DvC9XZhAJ0D7qnGF8ycODlGWUyXYjyXTHvrKKFIlDVdFSNJA5aDU8eRe5gE7jQhOHmcKYIXwdfSYEJZFmR9mo1mzazI8vCUMpscih50Hr8mKVh+niusES9RUTTxLyoMp3lqc9m3qmCetD4WPvWFSDb+qLjkvpFrdvXAXKozv6OEepkXmYZTaZ2uyErzmCXSL4pVXA050tYJ8iO1hksjtFksN9W+l7M/9z46nyUi9oFGgDNbS2i8FibEWGZqWf4dpm5lrhl+Qekmyb0cA83CsUhKqYcyC7a1Tude/P8g9qdpsumFw1lMOmvmTOIOcBrhh8+XApbZf9l7e3n/362t1pr/f0T9j2rMpfD5Kluve/fON/aa7+/bfNa0jM83agKaRVEsAUuFHwRYgue38AXkCIdHaGSnW3QWUBWxSI3992QQdmvtqwEmkdcXcS2BTlERd8Y/3h0fOd939191TyniwJMnmBt2leXerxip4O0JNtbaNwd+6E3wAMEkUctP8BH6LnbjG6bRQG849zy68kzXHLjBxIwGngtca2omsxAywFb20Y0n5lUUzCYeNAXzk10B8P4xFg0548HYT70BCnMUYTJgBaENAnxFca5mYoSD4IbvBt7QMg4Pfuo63Z+6R2fv5gcPPTo+6744Pv6BA1pVswMNkw7c3sRPHe8KeqRC/9fMJ08E3ZYbplKQamqDCh9Lx/8hxTFQjm9uLUKLHEoO/R1IG97UpSPmpFOxaiiota0qShvBLBlznpZVU7qQyVCg2W53Kw7c2yYbuSysFP7RAlBlMSfuhJQ13xlbOl/PDYyW88AWHSe8sKs16Xatzsea7lZdNVbwoc78p+157tMYBwSjg/KEBuMjcLhbzZax3MG6Z5FZh8EZqdTJgmdV2/frLQooRTOBu4iIbsBF+FDtXeAqbvhkh9BTpw0dQ+FTdhJlNzG4Lfy/W+0bipeRXTVQLey7GAK2J7vC50jgAXFAGX2Dd3oym1R+UUVSxaWE5aEcDGmV9Ls18SpwoL76l7zjVOanKNDVFOeOuyJuw2Cnnwm7GoDO3tC+ryaq7YVAhog/89FmiaSRjl0VtqCLfNC2W01jnkcSRqLqmGpfqTZM7Oz/fi5h50AHL1UkhRo9n+PsVuYXplZIH2pl7ejV7Kh+NGSmZWR2E0kDdg7gvijsgqHZU3BTKeYKY0jrD+cTlg8MGX4TIdnut4xkqSssJgZL5iTMeYfREBY2Qe0iBVANLX1ufbh++uzk6M2H+EP44dp2P4jA0HKvl5bSq9rbiPrjKhyJ7BnHW7+lid1utoZ3dVzY9VvW6b0NfNvoKyY4ohISpTA3YAY0SqO4FiDbFjOJGolZNh8UCyioRYctsmyKRakbdAIchWGinHQwgtj5RbWn4dtTh9e+zJYGGyTTmQ1Nbb4JloQsmqSoOMn2JMPKTFEUAGaOJtLZOVJfjShCBm0inR8jqfWiXkVmsyP7XcQQIcI0jb0rP5oBFRrM4pgiXIfmb8Dr8SVWE2utZ7f7nELxacgz5PzwnnckzpyOrmZWCqVBLYBuT5CTk7QSyZgvLEgSvu2VrmmeVZABRYMjfM7ZhqqecvFgl8UhzHuhM0KkftOAVY21oFlxCaCuxiZIvVeMhdZL8ZIZlJkvxWXzB21mmpm9SWZ4BNC6LVK/R7D4NzNiWYBJWCdliNiXVfDY96iRvTKmxTWyl7SsYL6zoGE52EKdVsRlz8d1BwtD4Umue+XdTxskXwW9Zn8uFC6ra1xUHLZaRG6vhNxeCTmTUGGpY+hST9kudsjPf6Gx2a3F2Sim0cM10FNZsjy3BWQFdwUyuGPl3y2xVGNS0xcrUhRg/fi++77rvOueHuwfHvxr/+zg+Khzsv/unayC9RqFun8cv3C6R6/en+4fvezmAN523x6f/uy8O9t/cXB4cPYzik0CZK2A+6Po/5C7dc4xuoyLK/wB1X/Lzv+3tncL+r9Wa63/+730fyvE9vtk7Z9Q3H2OT6PQ9W0+nJ5vNX0e02jQgRHX0EhJ6r6Kjc9VA/G/pRqg+4TUW0Gi0+2hVZVQ7YHC63HsIg6egjapFGPuwd5kDMbe4DJhMazYc0/cKURxxdlhFA6UotSzYFdMrUypZ8/T6NXMVYIbVmXJ/HKgByp53mTlpYCESjEb6mzmrzRzs6qqN3mVVxX5v5VqynQHAvEsxMuWIhIe7GZr+1P6gULvUmaJXrB45YMMDB2yd9tNOw+nD4mE28rD6f0h4FqtPFyxeQqssdqyF+FIswiay88fqsbnLWAqE8XOkoUq9bwA0uPn1n20TEDNiVHgf3Hp1gWDsIz9Zf0GTWIPNX4unok3+PYJPK5ypo3Yby/b5hWOBHK3lzXzChlcVqS8WaW8bNngUq63e3p6fOq8OH5/9GofWNf9ly+7J2cqa/vV8X90rO7/5jliwB8t/kdze7dlF+J/7Gyv+b/H5v/mc3Xqoa1DDt+OgzxXEgVXsCM3uAq2Z/cN5P6EplbMJXFISASI3ZGU3e+RP35JKpUuZKYrGRS2rkAe8EhPCuVQKCG3ysV0Y+Jfk2EVA6K3Ouxj9b3dp/hXgE1n57B1Sjj2Wk8x0FZQV2QiDs6i/mZll5FI4yIKoBMkEHulwnEX3JZlox1d7A5SCQnsbV18rDPCVyfCJ3IICs5Q9vBuNfS44CcORKkECLUYNYEk6aPKZgEwa7YKQgctBAPU9N0PBycn3VfAEPujEliJTtOzCjj214EdjIN+503HHh4kBfVfZ/7gUvQ2QDy3DIyLWIKH3zxCdk5uYFXlrsWh2MD0NMLfL+5xYpPnnVzeKYpul983AOwC3nCGG/ZqBgviagE3Hn50SU/IolmQVzO8tWqkc+NGtYD2zEsC1zzbFgEXLJmyWqiak5uzXKgajmee86nui3nPoDA1Q/XkTTTfRvVyhlvpQ0KXMWQOI+xVuNw+q5W41oIQeaf6mmS3NGRY6VaGDCt7XRGraARbS6LCJRpqsdhK1NRlqm8Bvlj/rfvcijzqVxVazNJ2ceXrjWGmXJIk6i0qqAslXC+fphYewBe6yokUjHQ4UWyaqRyBKGhJ39gvbwonL1r9iWo5mYSQp2IcTNdd5cZtELEJH/hXnsP0DdLsOJmdJ15q1fJGswp/6d202VW9ksv0bviNwdQ4Tmg4v9vPuE5Dv5RScp+LSdPKXYO6D0G3kM3mj2IpMirLSCk6ptM7T1RkGRxDazRFMmBaKNdck3BDgYuBIgcw/ZAuZWoKSuLha5wp3dhAIokcC+IjnWLd7yQdpe1ciCSMr+cpCieOAtOH0DKIFYB5DLzpiMu0dXFgVhRB1BLKBAqGTbC6dXYS3Eiv0/losCKvD472D50f3+8fHrw+eJnpwD9gd8wTW/J1GXYsVWdejvHxJQvB/3O6h9e2PqzydxX+fyd//9P29p695v9/J/3vJ1l9PohZ54rK40+z+lSMOB9KISzDQr37fh/rvbPrNUcDt9ncG24N3WfbO0Nvb2/v6dPzHff8qdt07S37fOfp3lZz+GzrfLe1PXB3d7y9wS789dyt5p41x2R0Oo7S6CJ2p2MzGpmumUyANv8BDUmNlYwpt1tzTSkfTX8+z4Ly87Rt0tBiscZNguVsxLiNniHvLlAAmUllLsSUYsNm5G3pPt/GcjX95MMcMjCrv3nHCZ9g+JcZ/ZUZ/EnnKfO5aT8t+MeXRoIvGvwZq1r7zbHoywxtNcMpxKBSlloBwvhkO0A2s8pMAT9fG72qmeBiE8FPMQ/8NNPAQi58Lqpn0PZvgZWfkV0SYuQCWs2xw8vuKi6a4UlZttRaLpNty43lpLArUdO7SC2IvwIsl9DPKQsEGHsXqaWGXsvsBRVLr1JzwZwkLHuwKAlbZTJGpaB5LB/TJZGicECLwshqRxRPnoj8FswcNkugjoFcIGzelB47vDk+fNU9cpApeqwzB8H/s1CeJEZGwQNLAEv4/1armdf/7zSba/7/q7X/cPilrg9tBCIFhKyURQcIUjKQnir8zmg8NPYwwgRaCGqutnyzXNF5jCI3q7jYNT+3q7sMsfrkjD6+mJUHv+ZQOYnJbc65aw+r1ZWMRBDtF7cKUU+FyqxDuOd/n/mWqKEABARTyGFPQkdlthy6U7sGXEivmfxiZGmFMIoUIJRb8IsAyxsZiDpy04JWs5mHKKvjfCMIXskMnemGw0KqveBgRGuHUq3CtieO+Yt72svjo7PT40Pn5HD/qPs1n6T/7z7/11Ttj7r/283tnVZx/1+f/6/9v79K/+8LivEEEjk0Bn2Wb6Sy7teZ76E3wxQdJwP30tN0dnOUcKtr8f6T/b5helQ4cauZn+39zZ++PtfvZQpMUtWVqiaXO/1uNec7/X6SR2uZG+vDMcOLNZJfrTtyqYtiMbiUFttdc0ItLpOi4yItqCUug6t4ni7TJpVGKxegOT+9Yo7MXlTLwI/ss9hXQFu4k3yCvcKs0XqV7FyjhjyzDSu5ktmt0LcWfdPhtqqas32OaumO96K8vPc9v9GzSPTYaEJ9hbcp3UL7Wa7l7JZWLd7T2tNc9zRn5oaql/mt2EG4vzn+/9Be5oyjv7+fuZhURWfz4lAXfc8La548zvUFX/QzZ5hX8TLnlwLP8TGHVYQUVd5N01FOF5nyBY3C8p7n98y1yB195MdJylitGgtVxtZhYuCR62zqBD6MFXyzd80nJhrb8D+05i+mM1zgFeajy4wCbYsvakLtDL0gdXF24VvBIRfgdQdhs56DlG7JBVB1yTQxsohaILyqDVBhsa0rVAR6VYNcrSLUhath10EfHP1qzVwVubwXYRnuDLDEEFUsVTTjipVzJjJUiyn+AEw9cWCiWe7F8+32ChnlLttT3PPLQTMvdFFxbqVZ2kTdFz1raobMvh8yeyGypsOs01Sn9vJBmu/9zyp1Hzz2XDyiPsolGUuqU95F98Bil2IpO0rLjjWlvaX0FGdRhIhNgz/zjt3mWdl/iWO3ogWhVEB+f/Dme5w3JbpH4aB+8F/dVw6/xszJu6t/3YpKof/TLEYf9/xvx242C/q/dfz3P5L936edBT6kTWC5po9PV/QmKVf1/e9QFOYPIIXQRBHS2593WojRd2/vPufMsPUpnuE5eUz5wOtFl+AOWN84FEz9QVzHs5NB9rygNDKnkyeUNVNNlmd2MAPjKEr1Y1GEz6z5lPCI/IRNQ9QpGXJp+gWoeyyiPI/1B6NQh2lpESKWjJu1PxAQmqNPnS2qegrrxso1xykedc6tcb5tOW1RdDkfuTgZLeDGA8nFmYST9z2qVfBSLlXgavr2QYRXSwS6xv3h9OqKO/sWhYH8vU1bF6qJP2/Rf3H71sUaZPtps2DmOm+GlRm2ZjqoshlnlGh9xVL/IkaoZWree9mfrqrdFX3E7/AqpXNLrESXLNQCxaN8m6jQonR+L9cy5VvR0zLzFm0vcRXlogyrOwtSJy8bLdUwa2rulUwWcznylotl1n7VexsrLvE5niM8zfWg+lzp6eT9i8ODl87Z+6Oj7uHafOOrlf+Yi92Nw7TGpNrG+2Qew/9ra8/Ox//a3tlrruW/R5T/LgLYJ8ptQb6wfMfUPkqUaboSG/ZWrFID/6vgXUqCVgI/4yIzRLTyCSeKIui0S6rxjDMhtoZJRSXhI/i+jLn4ZT/IxtER222LWDk8omOHcneqyEEVFidomF22opFA71QuvZtO4E7Oh6553Tav5WXVYv/sycCtihKVIaD9u5cvva/mLNPyqpnV80T1vDH7DoxW+TkFQ7M+qlgfVXzRo4oVZugnHVnwVbDiqYUOvT64+FoOLh7kMGLxoQebOvcOX/ZHPKQQ/B/JZo1k/CV4DGTq9nZ25vr/t7bz/J9tb+2t+b/H+P2fbzZnSbx57oebXnhlnrvJ2Ei81Kx3vVlkYqiPkesHxOp1PpHHM07fH50dvO12rD8jGum6YxlkVWPWpyakMBjL5EBBdJHIF+mgyd8zJpB/yAKNDYb8m2VgeKpez/zGrCdZAYq/kNnv/81E7xygJdObdByFW8CDfPfdxsnPG/O5XskYo0nVIPbSxJh2iAXeeChPpg1Y9Fo8ElZOgxJRGYL7ZGVLozobRHUgZ5Q0BuNJNKxMa2Yz2m02q8bJz8DvUXd8kzV0gEqakvOcv81TD24UDbrtps2Vyxs1qTisWubzzaF3tRnOgsBsPf+rLXv55Oez74+PTvbPvu8k8cAMo/FsmtVoYs6ufKCseHWDciF1I7uQuo2Xgtbr4yhJTVkN+EDNwNrAM0yceuBdeQGaGUXmc5xI/E5rEAouqELmX6Ey3mAcmX/+xnyuTA4ON/WHFnVZYVooIhLTlda0k4pa/nAM474o13Z++6zZNAI3YZffcmWmkv4dwAMLzc9pFilrVx4N0uISW07BnDqK3rZqSAVqJ7vcc4N92aBUdmUpavu+6ZhYcXEHMe5JI+ufx6c/dE9xGzp7/65zy0DvLN0CGzNQm1myjhZYyg1S6m+0l1oHzmkzy55vcswlNqW9rMGZVlavBQ0grwVTw767SVJv0r32oVN4Sw+P9185FB20bW58q+7tdHMnFFB2fvX+9JB0uYh8Ciy3oSpyW1UR5Y5PrmR2DjLewEO47BnJRKW3gQR6o7ZRDwbwP01hXrNXB/tvjo7fnR28fIfjcXoGqw4ot1kPYak0zeIyyNbov/8NM27m/c0MgYz6bj2Z+Nm3eWV0j15tgEg7v6tOu/uvfnZwWR2/P9sQfIyWSlzLBhGoL0Cdms+a96JOdJALm1QF+N3SDaOK1pncLWL/5MARGeiv9YnUDS9HXkjWoBnzyNogAj4UWOY0WoG2qcCCwNEMNG1J51bwBle6jw3pyNzgffKue/qTNrQfwsPjl/uHDsz/Tvn4fAgzX93OX5IP4UZpz6/VzF9a/5uk0fQLsf/L+P/mbisf/6u5t7v2///6+X9UxmCcX9THsAMvU6FSJtuu/mYOI9w3/SEZmGfk4c+YlZEy2suB1ydOn0Nq/D3lz2i7AKlaQgt86cN2UG+yNEvdHhkODnDWPX0rYOQWCYSUmIaJWR8pyA1UhBpDZPTKCeW7s+MToJY/vofO6r7iFHPD+J3WPzJCX4YCLFv/23be/s/estfr/6tZ/4K74Ot/BSbDMpT1rBGNTcXkTy5wU6wvflqN/IZgJgQ3Nw9J1aIFvN7/3Tjl5r8PTQWWrH97b283t/5bO1vr89+vf/9/cXCk7OeDIJoNR4Ebk2/68ZuOogrkKxalnOVqQ0Vv86XkyC1Njsw0jddQEWiVxnsMADdwBocgxcVeGt+YUK9IwGFxCZR34afj2XljEE2UflC7ZDP2As9NPKCbGJM73RxGH0PUa6hA9cAPZ9d1dzLc3caSURFowtLjpQk1oKSsvOd55xJX0++bf/2rwg7lpGEFtFrCJkGhbVJn4AOJnzCQyAkxaZg3mvN69XoY1d1ZGs2mQzSrrKMll1k6AOZzhmmxiKtUjpqK/KWDzOWf0avRtM2tZpWzk1ASsoMXMci/9W5kboiB6NXd+m/N+rP+tx8aMFhZ537AwdngLTL/zbUrtmADqwoXGgKUtkFlSjPYpP6SbIj0YuVpT9LAuemVKlWXbHMym3ftp2aT8Z6mKQR84kLZSNKAEJQNux5OJVhdFPYZxG26M+Phbn5Zyf6nZW8V6P+Wvab/j2n/k4v5osQCgc2A3yIhQ4JI3ZdVY1d6sbvI6u75AEAvxv4vl8EkjKa/xkk6u/p4ffNbHUZ0e2fXqrJzlRFaZE/9BuYdBD5d2MxKP4MvL+kLgyzoygTgIPaAZMCcnRrGYh+Ne1VQ+GLQgmDmyY4bDp1pFAR0Ib28355iioQe2edWCw4Jt5n5gQWlAnbsq8xzGL9vppPp5si99Mjxu1ZupQok0IZKka0GtL6jNLzCeDYmJXOfdFanDvtTMyfutcMugUj837xOS/HAz3q6ArhIZ86GQmkLFkffGnST23zzdBlWmUVp66i2+GMvCCJrlfDPWSQL4R7RUO5jYPbGLSMXrS6mg/xKIS8mz7E/L4WUUUzIgElsHLEbXniVnWY1Fxwk65mi+XSGT+kYPVSesNwutq+pwaHNOW/fPBNvvWLFACM5s3GYT61CXy0rYz580RqaObywKb9W2/7hf8r+L+4QefT9v1WU/7a21/HfHnn/L9/zN/J7/kZt4z476kbV+CJ7frZLM9HM8ROHe4vgfj3xQ8ASiO168dbJIrx+6tZIZ+ZCQFxlC2sWYRg5xeRbcfzeNjeiyw2NH2HRP1Fi9YHuOufE37ATsi/WUBEJVG0rO3Cv5luGDn/LsykR5zc0lm2jvcE5to8wBy827koK2FK7I2NBoE/wQrbE8YDnuOHXudI8mIWXIcjJIrDlvbtpPL+W91oFd5/FeG2UM15KX44F47XBGg/VBDywVpHp2mgDz7XBuwBSBMe1sQLHhffbPmi1iB2cUzF5AfCqNVvv3w+4/38R2X8l+X9nO3/+s72ztv98zP0/HSN19sOLsiAQU/RqLN2cccqI3fkf0flbN4REkDt/REnr9SwIyOxJJdqYwwH6iRh8uoZtSIQaih+MZbQyQaeDaHCJvsCido1D+CCiuw1S/wodWNl+iqKu8mmxlB5GIZ0+cSQ1JbceHwEroEtYvIxvO6atfdfKh5dK9qHGM5VGagSBbKe6YqF1tVCpaaDLCIV7KvMr4hqDyQWS7GxgKovVA1slUSJ85poFmBpMHYKm8UyQD++qJCiH5AiEDks1E6MXqkI0BguFVMDSLhOAEe1HF5BSOM/MSGtVQVTtdNE35EMQKE3AQgZBlHgy1D/NRNZwYBvp4khvKObcBaqi1TnXpcB61SVTCrOxttjVwhh92njYJeNRNhC2sluWpbfuchOMLegGme4lFX2x5jQdZfi2OL5CT1MfgPBQqZZ2/nq3/UPv/+L+3seW/5t7O82i/L+z3v8f41eyqQ+icORfSKcH5nH0rma+6r7ef394JnyQVFoqpg7t7N41RrvhivMcceV0O4dKD7mpRegAYiJqwOTjcudBJl9hEjok8ww91eWVpDkA19Jlfs0bsoLZGhRRtEY4GyxQKH+hS7X4s35zFtWwQrdZ03/Paqi/10KOVLBIgZueBW564bjpuQw33WlN/5XgtqT8ZAGHk+LGK5pqrLD+MdqQI1xrHpIILFn/9tZu3v9ra2untV7/fwT930Oc+X1hBSDNW8ZpJXSILeawg7oHB1aCM04nX1AXyOMf5c+ErKIOQ4+StlAnmAuX1rN4jJ06ixamBtMpxhebX7KiadT4PBllrL0oxFhNz8PvB8yiluXSYfgSfhjbbNiNZj59GA0SOgmlh1wij1qGyfwxB8BigWE6DyemJ9OlRW1TuXZJpt+tWdES+g9yJGw4OIkemANcRv93twr83+6a/j8q/Q9nk+kNErdwymjwycGhoLpk47dcF5RNH5GRfwEhmzAoJJunRDE3+02ASofO0ONaZM+JL85ZLkG1b25q5vU1kNhw2vDDIVCqpCI5rWrNpLtJO5A488N0ix9xu3HMcsAYDy4rFUDwF7O1s1sDdOIJPz4x7T3zW/z4xNyyqywJkLrXftKp29UGbF0YM5Kjf6oE+wf81D8N7BEoz72pwP+wR56+ecHJMF1bCnB6d1R4NDbZhnxgukRe/orUGiBKAMT1r/MhkhQDdzznuhE9bRb6v8482IWCKE4YFCy/8nESJyznAfSkPjR6P4Texwo1vmaq41NpAr8K/6rK3qqrIX5CNVaXedFO3HQw7ljZhLAUtcSyflxYfzpbcob+xAtxa0qWN4JmCZsPUsl1r0bIsu7TiMda/7z6fuh/uTKWyf/NfPyH5h781vT/MX49NgH63GSagm91zCQeGO5wGE2JcNV/XfNLX+tvejONIySNjTSaBL/L+gduL3//c3N37f/5SOv/fOYHw3pC/vt9QxiXmBRIDmTqaRpFQfK8wwJ0fRx7XmD1DZbpHFgBL8RAvgpog9JIBrcMo8dnV98gP0+MXz1HtqyTEJlwlxEuOWIGJjnCns5clfjXfe1eZXQ3MVlec+hNIjp7OoOd/savv90/MDVHFRMwMF8V8+ing1cH++bZtnndaliy9XVGDLGc552thm1bBsjXHpPxbzEmCya9PTiz7gyXrEOow25FG//n//3P/w0vzB/GkYsBWCfoKYBBcRvu5O9DN7y4hITG0Luy7vrQLgwjyKKcIRLgECyuKnnegbbbOyS4Wjx6AH7bYoK0Nb0BZNAFzztMMreIi5dvUz8Ioo/POzYHBzZoMIbUBsc49EejGXRagji3BVDshgn03oS+73CZ3UI2PSBODPDRx342uI2IhsUN6mpj+gZuLDSR2B7zvENzCB0trrHI1h5GiuvhrGko82cKswpNShoj4PP7Bky52GPTMR5AhjXJftCf8P9DvZkwAH3g6z+W0f+97a1C/NfdrfX5z2PK/3Pu/y21CpkfEHZ1Pe99tM1iWtZXUjtLffNKWmQ1hK3j0N3XDt4FkkTBFQi4DX7jX8/uG13YK7pHL/EyktIIg9DmuqwqU/DWUcGbsEiDMj/HucqVguL2kqyzesUe6t/rmpKf9g8PXuVuSxC1JplZ8dSYf5kBE7HR2sGBvcJhra1wxbgyzjUehr1tJmmMzZvCduGx16pZf25iGKz2CpfHtpSrYwvJ35UF0i9Vyo8yT41bVrM7a941sv5okb6e3c+ha9V1qxXZ2JL7UuYYAnGDDxZSaZ90NTCQ7BqAEYa2N0WtzaE/pDM+spsyb0Vhd5bQfwDTgYp+rY+Xm7aco+0RpDjcxsXR73Lm/YIFZ/Yuwmxnp7r0ulp9LbP6YPahCSBW0W6GX6Gpu0xpOITvVOZu1bTMJ+bu9nx/Km6Mo4wpMl63d/d1ssr11UrmO6ueKeUr2LMmfpJgYchv4gRb0TOLzG9p0Wvm1AsK8kPSTZUXVHA2mltyAVKafc8lU6Sbs+6KWbNG6GmrNuk8cMNLbhb+GU3SCIWwbH7yhFXMVAgqIrn7pJrOwmQ2xV3KG7KD/C9ZW0HYM7OBT600M7IH2qTWV/eM8ycXDp/DeZKrFaOVQ3G97+2FqDSYzfwy00NCvci/MAtkTnfVMHhB6DVvQVrYpXthTWJg0YNDbH110eJjV/tSRPDs7t5cpxbmgLqtiQLvSq9gXjC+eq2IYH5OzytTrcQMUTPxwaKWjYTeTYzKjmZB8OUWiDR2XLImFllB0rdZ6F65fuCeB96nGR4oCEo3C71veLhUJdPiPrr/ejIWTCDBiJiqgYG+ebXRU0i7/l3ZcDBxS0lUaQumbasZSxcLQj1ToDTij+7HLQ19jt7mAZSZhkkq4pKObps7ovZ3xYumBK+R8YY1LcXIrpfQuy+JZjEzzRhMZw4XsDJxnbHtCvggYnYZSjA1B9UuWRacYsBbOReAT7JJGg6lXhgDXXlVoIrx17OmS3nn8697zULOLkKgwlkoH708Pjo73X95VhJUncKgOA4q7ByHzA8dBzlmx+EO1Yx9Nv5j9D8wknj9j5BoH9QCZLH+x2617Hz8xy17Z33/6++l/4m9Fe/8WaYwKdOUGO++3wdBjW6bwYA6U/TKjK3/7jXrz9z6qH+7u333Z1igJ4f7L7vfHx++6p6i6nzh1W9c4qUgz7iimXqBR6pmB5pdqAQmqUQZkxp+4jDHUGDVxDZlitqamNQ2Sa9/ZxmKZKrcEEhosttjSy4aYtVjpTogqFbIY4frRGDj8IIyhQivJeuvBu5AZFDAMlN1+caJIHXsUzz0uCV8d1BpgvsmvrMWKAS8cDiLUc8LXUTdZ8kv9R3yB1TMCSmWdZJB4mud7Lvd+EYFvIgC6D4JyF7p7g3kqbZbKqzUPQnoJWo0lotu85BZ5t3twSYAOYVL2CXX6/FrLsXlNSwPuxuNG1jWearIkPGADj8YquBks+iT/5tXZyqMRnqdarcMl+9ygBl4CpzxH8LXB0f7h86P74Hlen3wcv/s4PiIx3+yCnf3EM4GDB6zCGVoYJKUVcMU0xyR8RItpRnYMP0SzKyIXtus4+WGvAC8lLNfLdaGoegoV6eU1QNvKjXxFAqvKSaSs6CnDbUUNqV6kv9gbmK0t5eBZVdmaY4GpMMTMPxCLnIs1gz2BQA3xqrJHML4KvtC1vvqB92E3+qv4CAgsg6VW4HPRzAOqb2rNo4IiSxYXEVp8tVG7dNNylhn9lSAvm4HnCnRCnhVbr2oVBPgc+/QZBdkwjINSD5W86hf+6WsZWGwBZepNk/SLayFn/ouw+lxZWxPSt81GQdo8ZPmEYJTXikBKSMfzB1DuDoiH3pNCm90vPDC2YTOS0vy5aMr0FAy7005jqOMEvduCfVdf86AMhFFgrPLTItdsKCHVkWbXcfFOpXqPO9OLtkRueavXFrZ4mZ7MEht6GSLZL/HLmXqZ9uCk5FvDiGucioMKWalUdxSU3rwuXDN3oy8XDHD6rfsqY6qas162W1oDNmSAhcgyo8IYZp7S9pqONVITgyhfntaHosy+XlaNvmph0vnu4aYMV84WLm5nrugELABZ1ljD61FjZhzn9v8O+zmXem2uAz73mXY9y6jePtbySWCiy6AW9aEe6K374W+cFfcKpVfvfvvh9xeGfkSzkLCsWwag6FhYhQr8K88rhlmh02X3k3bJA6912TLCr7U2BdaQcRr96zB2Btcwvpt+Kk3SSpVTauUw/pAGrdP1pdpdRMNYDGL5BgwT52+6Sd0h3tpl1JWRwgWRSZFaLeo+cyCzOEhhTFkY+KlCxRZWp8pYGo921S3FdRcmsbTUUSZteozCwHFe6UntJg8pMNcReaczDTqIhzE/dSaczCqE0M7sZf4yjOuShrUpjFgiVdTQ5BEqqLFC4j5RDOtdz8cnJx0X/GjOX9UAi+rUQjbJWB5ZDQMEkzginalbG8W2RSxgmNgcrNQCLBrq8qKZId/wi4QL1VWFvwcaFiCmQye70y9fGoE+2RwNUI8xXu8SIXc+CXyw6V6GVO9bLqLBDZGg5wq9bGmGtKdrLWg0KjCks7WrBIqMLPQaCSeGw/GlVg/+W5/SJ6wk+8Pybe9/fq/WODphlPv37aaNbRMYRjLe4TtfHOZ5IK+/NYiF32kgcD69qas/WWNz1reRxVCnvrdzdO5C42/81P39OD1z2td+/q3/q1/69/6t/79QX//H7Z1LjMAQAEA'

payload = base64.b64decode(PAYLOAD_B64)
assert hashlib.sha256(payload).hexdigest() == PAYLOAD_SHA256
manifest_bytes = json.dumps(MANIFEST, sort_keys=True, separators=(',', ':')).encode()
assert hashlib.sha256(manifest_bytes).hexdigest() == MANIFEST_SHA256

if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)
WORKSPACE.mkdir(parents=True)
with tarfile.open(fileobj=io.BytesIO(payload), mode='r:gz') as tf:
    tf.extractall(WORKSPACE, filter='data')

for item in MANIFEST:
    path = WORKSPACE / item['path']
    data = path.read_bytes()
    assert len(data) == item['bytes']
    assert hashlib.sha256(data).hexdigest() == item['sha256']

for name in ['outputs','metadata','evidence','logs','.runtime']:
    (WORKSPACE/name).mkdir(parents=True, exist_ok=True)
assert not any((WORKSPACE/'evidence').iterdir())
assert not any((WORKSPACE/'outputs').iterdir())
assert not any((WORKSPACE/'metadata').iterdir())
print(f'BOOTSTRAP_FILES={len(MANIFEST)}')
print('BOOTSTRAP_INTEGRITY=PASS')


## Step 3 — Static qualification before GPU load / Kiểm tra tĩnh trước khi nạp GPU

**English**

Run unit/contract tests, Python compilation, shell syntax validation, and notebook/source checks before loading the resident model.

**Tiếng Việt**

Chạy unit/contract test, compile Python, kiểm tra cú pháp shell và các static check trước khi nạp resident model.

### Expected evidence / Kết quả cần thấy

**English**

All test commands succeed and the cell ends with `STATIC_QUALIFICATION=PASS`.

**Tiếng Việt**

Tất cả lệnh test/validation đều thành công và cell kết thúc bằng `STATIC_QUALIFICATION=PASS`.


In [ ]:
import os, subprocess, sys
os.chdir(WORKSPACE)
env = dict(os.environ, PYTHONPATH=str(WORKSPACE/'src'))
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True, env=env)
subprocess.run([sys.executable, 'scripts/api_contract_acceptance.py'], check=True, env=env)
subprocess.run([sys.executable, '-m', 'compileall', '-q', 'src', 'scripts', 'tests'], check=True, env=env)
for script in (WORKSPACE/'scripts').glob('*.sh'):
    subprocess.run(['bash','-n',str(script)], check=True)
print('STATIC_QUALIFICATION=PASS')


## Step 4 — Start the resident BF16 T4×2 worker / Khởi động resident BF16 worker trên T4×2

**English**

Start the internal loopback worker immediately, stream `loading_model` progress while the BF16 pipeline loads in a background thread, and wait until the qualified component placement is resident across both T4 GPUs.

**Tiếng Việt**

Khởi động internal loopback worker ngay, hiển thị tiến độ `loading_model` trong khi BF16 pipeline được load bằng background thread, rồi chờ component placement đã qualification resident trên cả hai T4.

### Expected evidence / Kết quả cần thấy

**English**

Readiness JSON shows transformer on GPU0, text encoder + VAE on GPU1, dtype `bfloat16`, then `WORKER_READY=PASS`.

**Tiếng Việt**

Readiness JSON cho thấy transformer trên GPU0, text encoder + VAE trên GPU1, dtype `bfloat16`, sau đó là `WORKER_READY=PASS`.


In [ ]:
import os, secrets, subprocess, sys, time, urllib.error, urllib.request, json
RUNTIME = WORKSPACE/'.runtime'
TOKEN_FILE = RUNTIME/'api_token'
TOKEN_FILE.write_text(secrets.token_urlsafe(32))
os.chmod(TOKEN_FILE, 0o600)
os.environ['ZIMAGE_API_TOKEN'] = TOKEN_FILE.read_text().strip()
env = dict(os.environ, PYTHONPATH=str(WORKSPACE/'src'))
worker_log = open(WORKSPACE/'logs/worker.log','w')
worker = subprocess.Popen([sys.executable,'-m','uvicorn','zimage_server.worker_api:app','--host','127.0.0.1','--port','8101','--log-level','info'], cwd=WORKSPACE, env=env, stdout=worker_log, stderr=subprocess.STDOUT)
(RUNTIME/'worker.pid').write_text(str(worker.pid))

def _worker_diagnostics(reason):
    worker_log.flush()
    log_path = WORKSPACE/'logs/worker.log'
    tail = log_path.read_text(errors='replace')[-12000:] if log_path.exists() else '<worker.log missing>'
    print(f'WORKER_READY=FAIL reason={reason}')
    print('--- worker.log tail ---')
    print(tail)
    print('--- nvidia-smi ---')
    subprocess.run(['nvidia-smi'], check=False)

deadline=time.monotonic()+900
last_status=None
last_report=0.0
ready=None
while True:
    if worker.poll() is not None:
        _worker_diagnostics(f'worker exited with code {worker.returncode}')
        raise RuntimeError('worker readiness: process exited')
    try:
        with urllib.request.urlopen('http://127.0.0.1:8101/health', timeout=5) as r:
            health=json.load(r)
        status=health.get('status')
        assert status in {'starting','loading_model','ready','error'}, health
        now=time.monotonic()
        if status != last_status or now-last_report >= 15:
            elapsed=health.get('elapsed_seconds')
            elapsed_text='n/a' if elapsed is None else f'{elapsed:.1f}s'
            print(f'WORKER_STATUS={status} elapsed={elapsed_text}', flush=True)
            last_status=status; last_report=now
        if status == 'error':
            _worker_diagnostics(json.dumps(health.get('error')))
            raise RuntimeError(f'worker readiness: {health}')
        if status == 'ready':
            with urllib.request.urlopen('http://127.0.0.1:8101/ready', timeout=5) as r:
                ready=json.load(r)
            break
    except urllib.error.URLError:
        pass
    if time.monotonic()>deadline:
        _worker_diagnostics('900s timeout')
        raise TimeoutError('worker readiness')
    time.sleep(2)
expected={'transformer':0,'text_encoder':1,'vae':1}
assert ready['load']['device_map'] == expected, ready
assert ready['load']['dtype'] == 'bfloat16'
(WORKSPACE/'evidence/runtime-ready.json').write_text(json.dumps(ready, indent=2))
print(json.dumps(ready, indent=2))
print('WORKER_READY=PASS')


## Step 5 — Start the authenticated REST coordinator / Khởi động REST coordinator có xác thực

**English**

Start the CPU-side FastAPI coordinator on port 8090, validate the public health/root routes and authenticated readiness/info routes, and confirm only qualified profiles are exposed.

**Tiếng Việt**

Khởi động FastAPI coordinator phía CPU trên cổng 8090, kiểm tra health/root public cùng readiness/info có xác thực, và xác nhận chỉ expose các profile đã qualification.

### Expected evidence / Kết quả cần thấy

**English**

The runtime info JSON is printed and the cell ends with `COORDINATOR_READY=PASS`.

**Tiếng Việt**

Runtime info JSON được in ra và cell kết thúc bằng `COORDINATOR_READY=PASS`.


In [ ]:
coord_log = open(WORKSPACE/'logs/coordinator.log','w')
coord = subprocess.Popen([sys.executable,'-m','uvicorn','zimage_server.app:app','--host','127.0.0.1','--port','8090','--log-level','info'], cwd=WORKSPACE, env=env, stdout=coord_log, stderr=subprocess.STDOUT)
(RUNTIME/'coordinator.pid').write_text(str(coord.pid))

def get(path, auth=False):
    headers={'Authorization':'Bearer '+os.environ['ZIMAGE_API_TOKEN']} if auth else {}
    req=urllib.request.Request('http://127.0.0.1:8090'+path, headers=headers)
    with urllib.request.urlopen(req, timeout=10) as r:
        return r.status, json.load(r)

deadline=time.monotonic()+30
while True:
    try:
        status, health=get('/health'); break
    except Exception:
        if coord.poll() is not None: raise RuntimeError((WORKSPACE/'logs/coordinator.log').read_text()[-8000:])
        if time.monotonic()>deadline: raise TimeoutError('coordinator readiness')
        time.sleep(1)
assert status==200 and health['status']=='ok'
status, root=get('/')
assert root['type']=='rest-api'
status, ready=get('/ready', auth=True)
assert ready['ready'] is True
status, info=get('/v1/info', auth=True)
assert set(info['profiles']) == {'safe_512','high_768'}
assert 'high_1024' not in info['profiles']
print(json.dumps(info, indent=2))
print('COORDINATOR_READY=PASS')


## Step 6 — Run deterministic 512×512 acceptance / Chạy acceptance 512×512 xác định

**English**

Submit the golden `safe_512` request through the REST API and verify the output checksum against the accepted deterministic image.

**Tiếng Việt**

Gửi golden request `safe_512` qua REST API và đối chiếu checksum output với ảnh deterministic đã được chấp nhận.

### Expected evidence / Kết quả cần thấy

**English**

`GOLDEN_512_DETERMINISM=PASS`; expected PNG SHA-256 is `56e0fca007d3da945de77788b5ab8a0a131b58730d93b624ca65e7c6ca6ea307`.

**Tiếng Việt**

`GOLDEN_512_DETERMINISM=PASS`; SHA-256 PNG mong đợi là `56e0fca007d3da945de77788b5ab8a0a131b58730d93b624ca65e7c6ca6ea307`.


In [ ]:
from IPython.display import Image, Markdown, display

def _display_generation_event(event, label):
    png_bytes = (WORKSPACE / event['image_path']).read_bytes()
    assert png_bytes.startswith(b'\x89PNG\r\n\x1a\n')
    width = 512 if event['width'] <= 512 else 640
    display(Markdown(
        f"### {label} {event['index']}/{event['total']} — `{event['profile']}`\n"
        f"**Seed:** `{event['seed']}` · **Inference:** `{event['inference_seconds']:.2f} s` · "
        f"**SHA-256:** `{event['sha256'][:16]}…`"
    ))
    display(Image(data=png_bytes, width=width))

subprocess.run([sys.executable,'scripts/golden_rest_acceptance.py'], cwd=WORKSPACE, env=env, check=True)
golden=json.loads((WORKSPACE/'evidence/golden-512-seed42.json').read_text())
assert golden['sha256']=='56e0fca007d3da945de77788b5ab8a0a131b58730d93b624ca65e7c6ca6ea307'
golden_png=(WORKSPACE/'outputs/golden-512-seed42.png').read_bytes()
display(Markdown(
    f"### Golden deterministic output — `safe_512`\n"
    f"**Seed:** `42` · **Inference:** `{golden['inference_seconds']:.2f} s` · "
    f"**SHA-256:** `{golden['sha256'][:16]}…`"
))
display(Image(data=golden_png, width=512))
print('GOLDEN_512_DETERMINISM=PASS')


## Step 7 — Run five-job queue endurance / Chạy endurance queue 5 job

**English**

Exercise five sequential generation jobs through the bounded single-flight queue and record timing/queue evidence.

**Tiếng Việt**

Chạy năm generation job tuần tự qua bounded single-flight queue và ghi lại timing/queue evidence.

Each completed job is rendered immediately with its profile, seed, latency, and checksum so progress remains visible during the endurance run.

Mỗi job hoàn tất được render ngay cùng profile, seed, latency và checksum để người xem thấy tiến độ trong suốt endurance run.

### Expected evidence / Kết quả cần thấy

**English**

The endurance record is printed and the cell ends with `QUEUE_ENDURANCE=PASS`.

**Tiếng Việt**

Endurance record được in ra và cell kết thúc bằng `QUEUE_ENDURANCE=PASS`.


In [ ]:
live_env = dict(env)
live_env['ZIMAGE_NOTEBOOK_EVENTS'] = '1'
proc = subprocess.Popen(
    [sys.executable,'scripts/endurance_queue_acceptance.py'],
    cwd=WORKSPACE, env=live_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for raw_line in proc.stdout:
    line = raw_line.rstrip('\n')
    if line.startswith('NOTEBOOK_EVENT\t'):
        event = json.loads(line.split('\t', 1)[1])
        if event.get('event') == 'job_complete':
            _display_generation_event(event, 'Endurance image')
    else:
        print(line)
returncode = proc.wait()
if returncode:
    raise subprocess.CalledProcessError(returncode, proc.args)
endurance=json.loads((WORKSPACE/'evidence/endurance-5jobs.json').read_text())
print(json.dumps(endurance, indent=2))
print('QUEUE_ENDURANCE=PASS')


## Step 8 — Verify authentication and error boundaries / Xác minh authentication và error boundary

**English**

Exercise rejected requests for missing/invalid auth, invalid inputs, unsupported profiles/resolutions, queue pressure, worker unavailability, and premature image retrieval.

**Tiếng Việt**

Kiểm tra các request bị từ chối khi thiếu/sai auth, input không hợp lệ, profile/resolution không hỗ trợ, queue pressure, worker unavailable và lấy ảnh quá sớm.

### Expected evidence / Kết quả cần thấy

**English**

The boundary matrix is printed, 1024×1024 is rejected with HTTP 422, and the cell ends with `ERROR_BOUNDARIES=PASS`.

**Tiếng Việt**

Boundary matrix được in ra, 1024×1024 bị từ chối bằng HTTP 422 và cell kết thúc bằng `ERROR_BOUNDARIES=PASS`.


In [ ]:
subprocess.run([sys.executable,'scripts/error_boundary_acceptance.py'], cwd=WORKSPACE, env=env, check=True)
errors=json.loads((WORKSPACE/'evidence/error-boundary.json').read_text())
assert errors['checks']['unsupported_1024'][0] == 422
print(json.dumps(errors, indent=2))
print('ERROR_BOUNDARIES=PASS')


## Step 9 — Run mixed 512 → 768 → 512 qualification / Chạy qualification hỗn hợp 512 → 768 → 512

**English**

Prove that one resident pipeline can serve the qualified low/high/low profile sequence without replacing the model or creating concurrent GPU replicas.

**Tiếng Việt**

Chứng minh một resident pipeline có thể phục vụ chuỗi profile thấp/cao/thấp đã qualification mà không thay model hoặc tạo GPU replica chạy đồng thời.

Each completed 512/768/512 result is rendered immediately, turning the long qualification into a visible live sequence.

Mỗi kết quả 512/768/512 hoàn tất được render ngay, biến qualification dài thành một chuỗi live dễ theo dõi.

### Expected evidence / Kết quả cần thấy

**English**

The mixed-profile evidence is printed and the cell ends with `MIXED_PROFILE_QUALIFICATION=PASS`.

**Tiếng Việt**

Mixed-profile evidence được in ra và cell kết thúc bằng `MIXED_PROFILE_QUALIFICATION=PASS`.


In [ ]:
live_env = dict(env)
live_env['ZIMAGE_NOTEBOOK_EVENTS'] = '1'
proc = subprocess.Popen(
    [sys.executable,'scripts/mixed_profile_acceptance.py'],
    cwd=WORKSPACE, env=live_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for raw_line in proc.stdout:
    line = raw_line.rstrip('\n')
    if line.startswith('NOTEBOOK_EVENT\t'):
        event = json.loads(line.split('\t', 1)[1])
        if event.get('event') == 'mixed_complete':
            _display_generation_event(event, 'Mixed-profile image')
    else:
        print(line)
returncode = proc.wait()
if returncode:
    print(f'mixed_profile_acceptance.py exited {returncode}; qualifying captured evidence')
# The first 512 request can legitimately warm allocator state; qualify from captured results.
subprocess.run([sys.executable,'scripts/qualify_mixed_existing.py'], cwd=WORKSPACE, env=env, check=True)
mixed=json.loads((WORKSPACE/'evidence/mixed-512-768-512.json').read_text())
print(json.dumps(mixed, indent=2))
print('MIXED_PROFILE_QUALIFICATION=PASS')


## Step 10 — Optional public HTTPS transport check / Kiểm tra public HTTPS transport tùy chọn

**English**

Optionally expose the coordinator through a temporary Cloudflare Quick Tunnel and run an external REST acceptance check. This is transport evidence only, not runtime authority.

**Tiếng Việt**

Tùy chọn expose coordinator qua Cloudflare Quick Tunnel tạm thời và chạy external REST acceptance. Đây chỉ là transport evidence, không phải runtime authority.

### Expected evidence / Kết quả cần thấy

**English**

Either `PUBLIC_HTTPS_REST_ACCEPTANCE=PASS` or the explicit `PUBLIC_HTTPS_REST_ACCEPTANCE=SKIPPED` marker.

**Tiếng Việt**

Nhận một trong hai marker rõ ràng: `PUBLIC_HTTPS_REST_ACCEPTANCE=PASS` hoặc `PUBLIC_HTTPS_REST_ACCEPTANCE=SKIPPED`.


In [ ]:
RUN_PUBLIC_TUNNEL = False
public_evidence = WORKSPACE / 'evidence/public-tunnel-acceptance.json'

if not RUN_PUBLIC_TUNNEL:
    skipped = {
        'verdict': 'SKIPPED',
        'optional': True,
        'reason': 'Set RUN_PUBLIC_TUNNEL=True to run the external Quick Tunnel gate.'
    }
    public_evidence.write_text(json.dumps(skipped, indent=2))
    print(json.dumps(skipped, indent=2))
    print('PUBLIC_HTTPS_REST_ACCEPTANCE=SKIPPED')
else:
    CF_VERSION = '2026.10.0'
    CF = RUNTIME / 'cloudflared'
    if not CF.exists():
        url = f'https://github.com/cloudflare/cloudflared/releases/download/{CF_VERSION}/cloudflared-linux-amd64'
        urllib.request.urlretrieve(url, CF)
        os.chmod(CF, 0o755)
    tunnel_log = open(WORKSPACE / 'logs/cloudflared.log', 'w')
    tunnel = subprocess.Popen([
        str(CF), 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8090'
    ], stdout=tunnel_log, stderr=subprocess.STDOUT)
    (RUNTIME / 'tunnel.pid').write_text(str(tunnel.pid))
    import re
    deadline = time.monotonic() + 30
    public_url = None
    while time.monotonic() < deadline:
        time.sleep(1)
        text = (WORKSPACE / 'logs/cloudflared.log').read_text(errors='replace')
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', text)
        if m:
            public_url = m.group(0)
            break
    if public_url is None:
        raise RuntimeError('Quick Tunnel URL not established')
    (RUNTIME / 'tunnel_url').write_text(public_url)
    subprocess.run([sys.executable, 'scripts/public_tunnel_acceptance.py'], cwd=WORKSPACE, env=env, check=True)
    print('PUBLIC_HTTPS_REST_ACCEPTANCE=PASS')


## Step 11 — Finalize the qualification record / Hoàn tất qualification record

**English**

Aggregate the current-run evidence into the machine-readable summary and require a final PASS verdict. The local REST API remains available until you explicitly stop it.

**Tiếng Việt**

Tổng hợp evidence của run hiện tại thành summary machine-readable và bắt buộc final verdict là PASS. Local REST API vẫn hoạt động cho đến khi bạn chủ động dừng.

### Expected evidence / Kết quả cần thấy

**English**

`FINAL_QUALIFICATION=PASS`. To stop the services after review, run `bash scripts/stop.sh` from the materialized workspace.

**Tiếng Việt**

`FINAL_QUALIFICATION=PASS`. Sau khi review xong, có thể dừng service bằng `bash scripts/stop.sh` từ materialized workspace.


In [ ]:
subprocess.run([sys.executable,'scripts/finalize_evidence.py'], cwd=WORKSPACE, env=env, check=True)
subprocess.run([sys.executable,'scripts/verify_evidence.py'], cwd=WORKSPACE, env=env, check=True)
summary=json.loads((WORKSPACE/'evidence/qualification-summary.json').read_text())
assert summary['final_verdict']=='PASS', summary
notebook_identity={
    'api_only': True,
    'payload_sha256': PAYLOAD_SHA256,
    'payload_manifest_sha256': MANIFEST_SHA256,
    'model_variation':'dangkhoa2016/tongyi-mai-z-image-turbo/PyTorch/default/1',
    'upstream_revision':'f332072aa78be7aecdf3ee76d5c247082da564a6',
}
print(json.dumps({'qualification':summary,'notebook':notebook_identity}, indent=2))
print('FINAL_QUALIFICATION=PASS')
print('REST API remains on http://127.0.0.1:8090')
print('Bearer token remains private in .runtime/api_token; do not publish it.')
print('Stop services with: bash scripts/stop.sh')
